In [ ]:
# 🌡️ GRU Time Series Forecasting - Temperature Prediction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

# Create folders for saving images
os.makedirs("images", exist_ok=True)

# -------------------------------
# 📊 Load Dataset
# -------------------------------
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
df = pd.read_csv(url, parse_dates=['Date'], index_col='Date')

# -------------------------------
# 🔄 Data Preprocessing
# -------------------------------
data = df.values
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# -------------------------------
# 🔁 Create Sequences
# -------------------------------
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length), 0])
        y.append(data[i + seq_length, 0])
    return np.array(X), np.array(y)

sequence_length = 7
X, y = create_sequences(scaled_data, sequence_length)

# -------------------------------
# ✂️ Train-Test Split
# -------------------------------
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape for GRU [samples, time steps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# -------------------------------
# 🧠 Build GRU Model
# -------------------------------
model = Sequential([
    GRU(50, activation='relu', return_sequences=True, input_shape=(sequence_length, 1)),
    GRU(50, activation='relu'),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# -------------------------------
# 🚀 Train Model
# -------------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)

# -------------------------------
# 🔮 Predictions
# -------------------------------
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Inverse transform
train_predict = scaler.inverse_transform(train_predict)
y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))

test_predict = scaler.inverse_transform(test_predict)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

# -------------------------------
# 📏 Evaluation
# -------------------------------
train_rmse = np.sqrt(mean_squared_error(y_train_inv, train_predict))
test_rmse = np.sqrt(mean_squared_error(y_test_inv, test_predict))

print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")

# -------------------------------
# 📉 Plot Loss
# -------------------------------
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig("images/loss.png")
plt.show()

# -------------------------------
# 📊 Plot Predictions
# -------------------------------
plt.figure(figsize=(12, 6))

plt.plot(df.index[sequence_length:train_size+sequence_length], y_train_inv, label='Actual Train')
plt.plot(df.index[sequence_length:train_size+sequence_length], train_predict, label='Predicted Train')

plt.plot(df.index[train_size+sequence_length:], y_test_inv, label='Actual Test')
plt.plot(df.index[train_size+sequence_length:], test_predict, label='Predicted Test')

plt.title('GRU Temperature Prediction')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()

plt.savefig("images/prediction.png")
plt.show()

# -------------------------------
# 🔮 Forecast Next 30 Days
# -------------------------------
last_sequence = scaled_data[-sequence_length:]
future_predictions = []

for _ in range(30):
    next_day = model.predict(last_sequence.reshape(1, sequence_length, 1))
    future_predictions.append(next_day[0, 0])

    last_sequence = np.roll(last_sequence, -1)
    last_sequence[-1] = next_day

future_predictions = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

# -------------------------------
# 📈 Plot Forecast
# -------------------------------
plt.figure(figsize=(12, 6))

plt.plot(df.index[-100:], df.values[-100:], label='Historical Data')

future_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=30)
plt.plot(future_dates, future_predictions, label='30-Day Forecast')

plt.title('GRU 30-Day Temperature Forecast')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()

plt.savefig("images/forecast.png")
plt.show()